In [1]:
import sys
import os

# Aggiunge la cartella 'lib' al path
lib_path = os.path.abspath(os.path.join('..',))
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

In [2]:
print("\n".join(sys.path))


/home/mario/jobs/git_folders/AutoDFTB
/home/mario/miniconda3/envs/stm/lib/python310.zip
/home/mario/miniconda3/envs/stm/lib/python3.10
/home/mario/miniconda3/envs/stm/lib/python3.10/lib-dynload

/home/mario/miniconda3/envs/stm/lib/python3.10/site-packages


In [3]:
from pathlib import Path
import shutil
from tqdm import tqdm
from icecream import ic
import numpy as np
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde

from lib.electrodes_lib import (
    generate_electrode,
    adjust_outliers_atoms,
    check_interface_num_atoms,
    check_geometry_convergence,
    generate_electrode,
    adjust_outliers_atoms
)
from lib.dftb_lib import prepare_dftbplus_input
from lib.poscar_lib import xyz_to_poscar
from lib.utils_lib import (
    check_dir,
    check_file,
    read_from_xyz_file,
    write_to_xyz_file,
    launch_bin,
    get_cell_from_gen,
    move_xyz_to_origin,
    translate_xyz_file
)
from omegaconf import open_dict


/home/mario/miniconda3/envs/stm/lib/python3.10/site-packages/ase/structure.py:5: UserWarning: Moved to ase.build
  warnings.warn('Moved to ase.build')


## STEP 1: Fix xyz files
To ensure relevance to experimental conditions, structures with an in-plane carbon atom density of at least ~78% of pristine graphene were selected. From these, 50,000 structures were chosen for the reference dataset. To minimise edge effects in transport simulations, the largest defect in each flake was centred, and the flake was repositioned so that the atom with the lowest x and y coordinates was at (0, 0).

In [4]:
# This is the example of the /config/fix_xyz_dataset.yaml
package_path = Path('/home/mario/jobs/git_folders/AutoDFTB/example')
xyz_files_path = package_path.joinpath('test_xyz')
out_path = package_path.joinpath('test_batch/xyz_files_fixed')
nanoribbon_path = package_path.joinpath('fixed_xyz/nanoribbon.xyz')


cell = [34.43317005, 0.0, 0.0, 0.0, 34.08, 0.0, 0.0, 0.0, 10.0]
contact_vector = 2.459512143373903

In [35]:

def check_flake_orientation(X: np.ndarray, Y: np.ndarray) -> bool:
    zero_indices_X = np.where(X <= 0.35)[0]
    zero_indices_Y = np.where(Y <= 0.35)[0]

    common_elements = sum(element in zero_indices_X for element in zero_indices_Y)

    if common_elements == 1:
        return True
    else:
        return False


def mirrors_xyz_structure(X: np.ndarray) -> np.ndarray:
    X = -X
    X -= np.min(X)
    return X


def custom_adjust_outliers_atoms(
    X: np.ndarray,
    Y: np.ndarray,
    cell_x: float = 34.43317005,
    cell_y: float = 34.08,
    delta_x: float = 0.4,
    delta_y: float = 0.4,
):

    new_X = []
    new_Y = []
    for x, y in zip(X, Y):
        if cell_x - x < delta_x:
            x -= cell_x
        if cell_y - y < delta_y:
            y -= cell_y
        new_X.append(x)
        new_Y.append(y)

    return np.array(new_X), np.array(new_Y)


def main(xyz_files_path, out_path, cell, contact_vector, nanoribbon_path):
    xyz_files_path = Path(xyz_files_path)
    out_path = Path(out_path)
    cell = list(cell)
    contact_vector = float(contact_vector)
    nanoribbon_path = Path(nanoribbon_path)

    check_dir(xyz_files_path)
    check_file(nanoribbon_path)
    out_path.mkdir(exist_ok=True, parents=True)

    files = [f for f in xyz_files_path.iterdir() if f.suffix.lower() == ".xyz"]

    for file in tqdm(files):
        atoms, X, Y, Z = read_from_xyz_file(file)
        try:
            kde = gaussian_kde(X)
            x_grid = np.linspace(min(X), max(X), 5000)
            kde_data = kde.evaluate(x_grid)
            peaks, _ = find_peaks(-kde_data)
            peaks = sorted(peaks, key=lambda x: kde_data[x])

            cell_x = float(cell[0])
            cell_y = float(cell[4])
            desired_centroid = cell_x / 2
            offset = x_grid[peaks[0]] - desired_centroid

        except Exception as e:
            # print(e)
            continue

        if offset >= 0:
            X[X < offset] += cell_x
        else:
            X[X > (cell_x - abs(offset))] -= cell_x
        X -= np.min(X)
        Y -= np.min(Y)
        X, Y = custom_adjust_outliers_atoms(X, Y, cell_x=cell_x, cell_y=cell_y)
        if not check_flake_orientation(X, Y):
            X = mirrors_xyz_structure(X)
        if not check_flake_orientation(X, Y):
            continue
        write_to_xyz_file(out_path.joinpath(file.name), atoms, X, Y, Z)

        generate_electrode(
            file_path=out_path.joinpath(file.name),
            out_path=out_path.joinpath(file.name),
            electrode_path=nanoribbon_path,
            cell=cell,
            contact_vector=contact_vector,
        )
        adjust_outliers_atoms(
            out_path.joinpath(file.name),
            out_path.joinpath(file.name),
            cell_x=cell_x + (2 * contact_vector),
            cell_y=cell_y,
        )


if __name__ == "__main__":
    main(xyz_files_path, out_path, cell, contact_vector, nanoribbon_path)


100%|██████████| 2/2 [00:00<00:00, 20.60it/s]


## STEP 2: Geometry optimization
Now we have to optimize the geometry of the fixed flakes, obtained from the previous point. To do so we firstly get the standard cell, by manually optimizing the geometry and the lattice of a perfect graphene flakes with the same dimension of the flakes in the dataset.

In [39]:
def optimize_geom(args):
    """
    Performs full geometry optimization workflow for a single xyz file.

    This includes:
    - Preparing the input (POSCAR, DFTB+ input)
    - Running DFTB+
    - Post-processing output files
    - Geometry validation and cleanup
    """
        
    dftb_bin_path = Path(args.dftb_bin_path)
    working_dir = Path(args.working_dir)
    file = Path(args.file)
    fixed_path = Path(args.xyz_dir_fixed)
    fixed_path.mkdir(exist_ok=True, parents=True)
    box_size = list(args.box_size)
    bond_lenght = args.bond_lenght
    delta_x = args.delta_x
    delta_y = args.delta_y
    delta_interface = args.delta_interface

    shutil.copy(file, working_dir.joinpath(file.name))
    local_path = (Path(args.package_path).joinpath(args.slurm_output, file.stem))
    local_path.mkdir(exist_ok=True, parents=True)
    print(local_path)
    xyz_to_poscar(
        working_dir.joinpath(file.name),
        working_dir.joinpath(f"{file.stem}.POSCAR"),
        default_box_size=box_size,
    )
    prepare_dftbplus_input(args, working_dir.joinpath(f"{file.stem}.POSCAR"))
    
    if args.scheduler == "slurm":
        write_out_file = None
    elif args.scheduler == "local":
        write_out_file = str(local_path.joinpath(file.stem)) + ".out"
    else:
        raise ValueError(f"Scheduler {args.scheduler} not supported")
    
    launch_bin(dftb_bin_path, working_dir, verbose=args.verbose, write_out_file=write_out_file)
    shutil.copy(
        working_dir.joinpath(f"opt_{file.stem}.xyz"),
        fixed_path.joinpath(f"{file.stem}_opt.xyz"),
    )
    get_cell_from_gen(
        working_dir.joinpath(f"opt_{file.stem}.gen"),
        json_output_path=fixed_path.joinpath(f"{file.stem}_opt.json"),
        custom_name=file.stem,
    )
    # === Check convergence === #
    #print(Path(args.package_path).joinpath(args.slurm_output, file.stem))
    if not check_geometry_convergence(
        Path(args.package_path).joinpath(args.slurm_output, file.stem)
    ):
        print(f"Warning, Geometry did NOT converge for {file.stem}!")
        if args.remove_if_not_converged:
            os.remove(str(fixed_path.joinpath(file.stem + "_opt.xyz")))
            os.remove(str(fixed_path.joinpath(f"{file.stem}_opt.json")))
            shutil.rmtree(working_dir)
            return

    translate_xyz_file(fixed_path.joinpath(f"{file.stem}_opt.xyz"), z_offset=-5.0)
    move_xyz_to_origin(fixed_path.joinpath(f"{file.stem}_opt.xyz"))

    adjust_outliers_atoms(
        fixed_path.joinpath(f"{file.stem}_opt.xyz"),
        fixed_path.joinpath(f"{file.stem}_opt.xyz"),
        cell_x=box_size[0],
        cell_y=box_size[4],
        delta_x=delta_x,
        delta_y=delta_y,
    )

    if not check_interface_num_atoms(
        fixed_path.joinpath(f"{file.stem}_opt.xyz"),
        cell_y=box_size[4],
        delta=delta_interface,
        bond_lenght=bond_lenght,
    ):
        os.remove(str(fixed_path.joinpath(f"{file.stem}_opt.xyz")))
        os.remove(str(fixed_path.joinpath(f"{file.stem}_opt.json")))
        shutil.rmtree(working_dir)
        return

    shutil.rmtree(working_dir)


In [40]:
def main(args):
    """
    Main entry point for geometry optimization.

    Depending on the scheduler type (slurm/local), distributes the xyz files 
    for DFTB+ optimization either locally or through a SLURM job scheduler.
    """
    
    if args.verbose:
        ic.enable()
    else:
        ic.disable()

    # === Get hydra config paths === #
    xyz_dir = Path(args.xyz_dir)
    check_dir(xyz_dir)
    working_dir = Path(args.working_dir)
    out_path = Path(args.xyz_dir_fixed)
    
    # === Filter already processed files === #
    if out_path.is_dir():
        already_done = [
            f.stem[:-4] for f in out_path.iterdir() if f.suffix.lower() == ".xyz"
        ]
        files = [
            f
            for f in xyz_dir.iterdir()
            if (f.suffix.lower() == ".xyz" and f.stem not in already_done)
        ]
    else:
        already_done = []
        files = [f for f in xyz_dir.iterdir() if f.suffix.lower() == ".xyz"]
        
    # === Choose execution strategy === #
    run_local(files, args, working_dir)



def run_local(files, args, working_dir):
    """
    Processes all files locally by calling `optimize_geom()` sequentially.
    """
    for file in tqdm(files):
        local_working_dir = working_dir.joinpath(f"tmp_{file.stem}")
        local_working_dir.mkdir(exist_ok=True, parents=True)
        with open_dict(args):
            args.working_dir = str(local_working_dir)
            args.file = str(file)
        print(f"Processing {file.name}...")
        optimize_geom(args)
        

In [44]:
from omegaconf import OmegaConf

args = OmegaConf.create({
    'package_path': str(Path('/home/mario/jobs/git_folders/AutoDFTB/example')),
    'dftb_bin_path': str(Path('/home/mario/app/dftbplus/bin/dftb+')),
    'slakos': str(Path('/home/mario/app/dftbplus/slakos/pbc-0-3')),
    'working_dir': str(Path('/home/mario/jobs/git_folders/AutoDFTB/example/tmp')),
    'xyz_dir': str(Path('/home/mario/jobs/git_folders/AutoDFTB/example/test_batch/xyz_files_fixed')),
    'xyz_dir_fixed': str(Path('/home/mario/jobs/git_folders/AutoDFTB/example/test_batch/dftb/xyz_files_fixed_opt')),
    'periodic': True,
    'box_size': [39.53476932, 0.0, 0.0, 0.0, 34.27629786, 0.0, 0.0, 0.0, 10.0],
    'bond_lenght': 1.42,
    'delta_x': 0.4,
    'delta_y': 0.4,
    'delta_interface': 1.88,
    'optimize_geometry': True,
    'optimize_lattice': False,
    'max_steps': 200,
    'remove_if_not_converged': True,
    'verbose': True,
    'slurm_output': 'outputs/slurm_out',
    'scheduler': 'local',
})  

main(args)


  0%|          | 0/2 [00:00<?, ?it/s]

Processing graphene_3467.xyz...
/home/mario/jobs/git_folders/AutoDFTB/example/outputs/slurm_out/graphene_3467
Warining truncateskrange block skipped!


 50%|█████     | 1/2 [01:45<01:45, 105.67s/it]

Processing graphene_1982.xyz...
/home/mario/jobs/git_folders/AutoDFTB/example/outputs/slurm_out/graphene_1982
Warining truncateskrange block skipped!


100%|██████████| 2/2 [03:03<00:00, 91.93s/it] 
